import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

DATA_PATH = "../data/raw/smart_home_energy_consumption_large.csv"

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
df.head()

In [ ]:
data_dictionary = pd.DataFrame({
    "Column": [
        "Home ID", "Appliance Type", "Energy Consumption (kWh)",
        "Time", "Date", "Outdoor Temperature (°C)",
        "Season", "Household Size",
    ],
    "Data Type": [
        "int64", "string (categorical)", "float64",
        "string (HH:MM)", "string (YYYY-MM-DD)", "float64",
        "string (categorical)", "int64",
    ],
    "Unit": ["-", "-", "kWh", "24-hour clock", "date", "°C", "-", "persons"],
    "Role": [
        "Identifier (not used as a feature)", "Input feature", "TARGET",
        "Input feature (hour extracted)", "Input feature (month, weekday extracted)",
        "Input feature", "Input feature", "Input feature",
    ],
})
data_dictionary

In [ ]:
df.info()
df.describe()

In [ ]:
# Rename columns to simple, code-friendly names
df = df.rename(columns={
    "Home ID": "home_id",
    "Appliance Type": "appliance_type",
    "Energy Consumption (kWh)": "energy_kwh",
    "Time": "time",
    "Date": "date",
    "Outdoor Temperature (°C)": "outdoor_temp_c",
    "Season": "season",
    "Household Size": "household_size",
})

# Duplicates and missing values
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Missing values:", df.isnull().sum().sum())

# Date and time features
df["date"] = pd.to_datetime(df["date"])
df["hour"] = pd.to_datetime(df["time"], format="%H:%M").dt.hour
df["month"] = df["date"].dt.month
df["day_of_week"] = df["date"].dt.dayofweek      # 0 = Monday
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

df.head()

In [ ]:
# 1) Does the Season label match the month?
print(pd.crosstab(df["month"], df["season"]))

# 2) Consumption by appliance (high values for heater/AC are expected, not errors)
print(df.groupby("appliance_type")["energy_kwh"].describe().round(2))

In [ ]:
df.to_csv("../data/processed/cleaned_energy_data.csv", index=False)
print("Saved:", df.shape)